## Grokking Hacking 3

- Ok here's my swaggy idea right now -> can I just inspect various parts of the model, show shit going in circles/waves, compare to sines/cosines and make point?
- Big push here for sure is finding minimal path through this stuff.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.11.0

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}

if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
from functools import partial
from pathlib import Path

import einops
import numpy as np
import torch as t
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from jaxtyping import Float
from torch import Tensor
from tqdm import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.utils import to_numpy
import matplotlib.pyplot as plt
from ipywidgets import interact

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part52_grokking_and_modular_arithmetic"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

grokking_root = section_dir / "Grokking"
saved_runs_root = grokking_root / "saved_runs"

import part52_grokking_and_modular_arithmetic.tests as tests
import part52_grokking_and_modular_arithmetic.utils as utils

device = t.device("cuda" if t.cuda.is_available() else "cpu")

t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

In [ ]:
p = 113

cfg = HookedTransformerConfig(
    n_layers=1,
    d_vocab=p + 1,
    d_model=128,
    d_mlp=4 * 128,
    n_heads=4,
    d_head=128 // 4,
    n_ctx=3,
    act_fn="relu",
    normalization_type=None,
    device=device,
)

model = HookedTransformer(cfg)

In [ ]:
REPO_ID = "callummcdougall/grokking_full_run_data"
FILENAME = "full_run_data.pth"

local_dir = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

full_run_data = t.load(local_dir, weights_only=True)
state_dict = full_run_data["state_dicts"][400]

model = utils.load_in_state_dict(model, state_dict)

In [ ]:
# Get all data and labels, and cache activations
all_data = t.tensor([(i, j, p) for i in range(p) for j in range(p)]).to(device)
labels = t.tensor([utils.target_fn(i, j) for i, j, _ in all_data]).to(device)
original_logits, cache = model.run_with_cache(all_data)

# Final position only, also remove the logits for `=`
original_logits = original_logits[:, -1, :-1]

# Get cross entropy loss
original_loss = utils.cross_entropy_high_precision(original_logits, labels)
print(f"Original loss: {original_loss.item():.3e}")

In [ ]:
cache["pre", 0].shape

In [ ]:
neuron_acts_pre = cache["pre", 0][:, -1]

In [ ]:
neuron_acts_pre.shape

In [ ]:
fig=plt.figure(0, (12,12))
for i in range(12):
    for j in range(12):
        fig.add_subplot(12,12,12*i+j+1)
        plt.scatter(neuron_acts_pre[:113, i].detach().cpu().numpy(), neuron_acts_pre[:113, j].detach().cpu().numpy(), c=range(113), s=1)
        plt.axis('off')

Ok finding obvoius sinusoids in the start of the model is fuzzy - what about interpreting how an actual addition problem gets soloved
from the persprctive of this mid MLP layer? Possible? Can we see the angle addition happening??

In [ ]:
fig=plt.figure(0, (12,12))
for i in range(12):
    for j in range(12):
        fig.add_subplot(12,12,12*i+j+1)
        plt.scatter(neuron_acts_pre[50*113:51*113, i].detach().cpu().numpy(), neuron_acts_pre[50*113:51*113, j].detach().cpu().numpy(), c=range(1*113), s=1)
        plt.axis('off')

Ok, maybe a rigorous-ish thing to do this first half of the day is try to answer: 
1. Exactly what trig equations of what variables approximate the patterns I'm seeing here in the MLP? Hmm do "quadratic terms" fall out here somehow? What would be cool. 
2. Once I have them, how does the network approximate these? Is the functional form helpful? 
3. Then going downstream, how do these get turned into a final answer? Is the functional form helpful?

Hmmm, smoothly changing a and be and seeing the shapes the get swept out?

In [ ]:
m=cache['blocks.0.mlp.hook_pre'].detach().cpu().numpy()
mr = einops.rearrange(cache['blocks.0.mlp.hook_pre'], "(x y) ... -> x y ...", x=p).detach().cpu().numpy()

In [ ]:
fig=plt.figure(0, (12,12))
for i in range(12):
    for j in range(12):
        fig.add_subplot(12,12,12*i+j+1)
        plt.scatter(m[50*113:51*113, 2, i], m[50*113:51*113, 2, j], c=range(1*113), s=1)
        plt.axis('off')

In [ ]:
def viz_neuron(neuron_idx):
    plt.plot(mr[:p, 0, 2, neuron_idx])
    plt.plot(mr[0, :p, 2, neuron_idx])

In [ ]:
interact(viz_neuron, neuron_idx=(0, 512, 1))

What function(s) of a and b exactly are the two different types of neurons here implementing?

In [ ]:
mr[:p, 0, 2, 1].shape

In [ ]:
def approximate_fourier(x, num_freqs, keep_dc=True):

    x = np.asarray(x)
    N = x.shape[0]

    # Real FFT
    coeffs = np.fft.rfft(x)          # shape: (N//2 + 1,)
    n_bins = coeffs.shape[0]

    # DC is coeffs[0], non-DC frequencies are coeffs[1:]
    non_dc = coeffs[1:]
    n_non_dc = non_dc.shape[0]

    num_freqs = min(num_freqs, n_non_dc)

    # Find indices (in 1..N//2) of the largest-magnitude non-DC coefficients
    mags = np.abs(non_dc)
    top_idx = np.argsort(mags)[::-1][:num_freqs]   # indices in 0..n_non_dc-1
    top_idx = top_idx + 1                          # shift to 1..N//2

    # Build truncated coefficient array
    coeffs_trunc = np.zeros_like(coeffs, dtype=complex)
    if keep_dc:
        coeffs_trunc[0] = coeffs[0]
    coeffs_trunc[top_idx] = coeffs[top_idx]

    return coeffs_trunc, N


def reconstruct_from_coeffs(coeffs_trunc, N):
    return np.fft.irfft(coeffs_trunc, n=N)


In [ ]:
coeffs_trunc, N_rec = approximate_fourier(mr[:p, 0, 2, 1], num_freqs=1)
x_approx = reconstruct_from_coeffs(coeffs_trunc, N_rec)

In [ ]:
plt.plot(mr[:p, 0, 2, 1])
plt.plot(x_approx)

In [ ]:
coeffs_trunc

In [ ]:
coeffs_trunc.shape

- Plumbing to get to purse sine/cosing is a bit annoying -> i think doing one or two manually is probably ok.
- It probably will be worth figuring out if the "interferency" ones can be approximated as a sum of sine/cosine
- Becuase, I kinda suspect that we have to actually do a product, and that these are the "quadratic" terms that Neel talks about

In [ ]:
np.where(coeffs_trunc.real!=0)

In [ ]:
def my_fit(n):
    return coeffs_trunc[0].real/p+\
           (2.0 / p)*(coeffs_trunc[14].real*np.cos(2.0 * np.pi * 14 * n / p)-\
                      coeffs_trunc[14].imag*np.sin(2.0 * np.pi * 14 * n / p))

In [ ]:
plt.plot(mr[:p, 0, 2, 1])
plt.plot(my_fit(np.arange(p)))


- Nice that looks pretty close!
- Now, hos does this connect to the summation?
- If the sum is alaredy like incorperated into this neuron, our result should be equivariant in some sense
- I mean it should just be a funtion of the sum and not depend on a or b explicitly. What's a cool way to show this?

In [ ]:
mr.shape

In [ ]:
plt.imshow(mr[:,:,2,1])

- This might be a cool 3d manim surface, especialy if we build it up from 2d
- All the pairs that add to the same value should result in the same neuron output. e.g. (0, 10), (1, 9), (2, 8). ..
- 

In [ ]:
all_data.shape

In [ ]:
inputs_arranged_by_sum=[]
for s in range(p):
    # inputs_arranged_by_sum.append([])
    for i in range(p):
        inputs_arranged_by_sum.append([i, (s-i)%p, p])
inputs_arranged_by_sum=t.tensor(inputs_arranged_by_sum).to(device)

In [ ]:
inputs_arranged_by_sum.shape

In [ ]:
logits, cache_by_sum = model.run_with_cache(inputs_arranged_by_sum)

m_by_sum=cache_by_sum['blocks.0.mlp.hook_pre'].detach().cpu().numpy()
mr_by_sum = einops.rearrange(cache_by_sum['blocks.0.mlp.hook_pre'], "(x y) ... -> x y ...", x=p).detach().cpu().numpy()

In [ ]:
mr_by_sum.shape

In [ ]:
plt.imshow(mr_by_sum[:,:,2,1])

In [ ]:
# inputs_arranged_by_sum[:p]

In [ ]:
plt.plot(mr_by_sum[0, :, 2, 1])

- Hmm ok shouldn't this be constant? Is it not taking a sum yet? Or maybe something a little different is going on?
- What 2D function is this neuron actually implementhing then? Maybe that's the better question?
- I think I'm kinda maybe recreating what Neel did, but I think that's fine/good!

In [ ]:
plt.imshow(mr[:,:,2,1])

- Hmm yeah what is the 2D/2 input function here? 
- ...taking a slice where the sum is constant....

In [ ]:
plt.plot(mr[:p, 10, 2, 1])
plt.plot(mr[10, :p, 2, 1])

Having trouble thinking of what equation this is -> tons of structure here -> feels like it's "cosine of x OR y" lol. 

---

In [ ]:
plt.plot(mr[:p, 10, 2, 1])
plt.plot(mr[10, :p, 2, 1])

In [ ]:
plt.plot(mr[:p, 5, 2, 1])
plt.plot(mr[5, :p, 2, 1])

In [ ]:
plt.plot(mr[:p, 0, 2, 4])
plt.plot(mr[0, :p, 2, 4])

In [ ]:
m.shape

In [ ]:
plt.imshow(m[:3*p, 2, :])

In [ ]:
plt.plot(m[:2*p, 2, 1])

- Seems like one pretty focused frequency!
- Are these the ones that don't actually matter though and the product of two frequencies is more relevant?

In [ ]:
plt.plot(mr[:p, 0, 2, 4])

That's a nice classic interference pattern.

In [ ]:
plt.plot(mr[:p, 0, 2, 4])
plt.plot(mr[0, :p, 2, 4])

- Woah that's kind surprising right? Same exact thing happens when moving a or b?
- This neuron has incredibly closely approximate the function cos...

In [ ]:
plt.plot(mr[:p, 1, 2, 95])
plt.plot(mr[1, :p, 2, 95])

- Ok are we actually seeing a result of learned commutivity of addition?
- Is that why we get very similiar outputs when varying a or b?

In [ ]:
# mr[:p, 0, 2, 2]-mr[0, :p, 2, 2] # Not identical, but very close. 

In [ ]:
mr.shape

---

- Ok how does this actually work lol?
- As we increase one of our inputs, we go around the circle, well sort off.
- Hmm i feel like I'm close to seeing the rotation, but not quite there...

----

- Ok let me look at the embeddings more closely here.
- There should be some signal there!

In [ ]:
# Helper variables
W_O = model.W_O[0]
W_K = model.W_K[0]
W_Q = model.W_Q[0]
W_V = model.W_V[0]
W_in = model.W_in[0]
W_out = model.W_out[0]
W_pos = model.W_pos
W_E = model.W_E[:-1]
final_pos_resid_initial = model.W_E[-1] + W_pos[2]
W_U = model.W_U[:, :-1]

print("W_O  ", tuple(W_O.shape))
print("W_K  ", tuple(W_K.shape))
print("W_Q  ", tuple(W_Q.shape))
print("W_V  ", tuple(W_V.shape))
print("W_in ", tuple(W_in.shape))
print("W_out", tuple(W_out.shape))
print("W_pos", tuple(W_pos.shape))
print("W_E  ", tuple(W_E.shape))
print("W_U  ", tuple(W_U.shape))

In [ ]:
plt.imshow(W_E.detach().cpu().numpy())

In [ ]:
cache['hook_embed'].shape

In [ ]:
plt.imshow(cache['hook_embed'][:p, 1, :].detach().cpu().numpy())

Ok right yeah the embedding is the matrix here. make sense. 

In [ ]:
plt.plot(cache['hook_embed'][:p, 1, 17].detach().cpu().numpy())

In [ ]:
plt.plot(cache['hook_embed'][50, 1, :].detach().cpu().numpy())

- Ok so the fourier transform is pretty clean, but the individual slices are not
- Guessing that's becuase the embedding matrix is spreading out frequencies across embedding vectors
- I'm curious how Nanda ablates or restricts this part of the model in freuency space -> this could be pretty telling
- Buuut, from the perspective of "seeing this thing work", I don't think this is quite the right starting point!
- Ok, so if not here, then where?

In [ ]:
def viz_5(x=0, y=0):
    input_1=t.tensor([[x, y, 113]]).to(device)
    out_1, cache=model.run_with_cache(input_1)    
    fig=plt.figure(0, (6, 4))
    with t.no_grad():
        manual_unembed=model.unembed(cache['hook_embed'].detach())
        fig.add_subplot(4, 1, 1); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
        manual_unembed=model.unembed(cache['blocks.0.hook_attn_out'].detach())
        fig.add_subplot(4, 1, 2); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
        manual_unembed=model.unembed(cache['blocks.0.hook_mlp_out'].detach())
        fig.add_subplot(4, 1, 3); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
        manual_unembed=model.unembed(cache['blocks.0.hook_resid_post'].detach())
        fig.add_subplot(4, 1, 4); plt.plot(manual_unembed[0, -1, :-1].detach().cpu().numpy())
    plt.show()

In [ ]:
interact(viz_5, x=(0, 113, 1), y=(0, 113, 1))

- Ok roughly, after just the embedding layer, there should be some sin/cos components?
- I'm not really seeing that, not quite sure what I'm missing.
- Ok maybe the output of this W_neur thing would be more telling/helpful? I think that's before attention -> let me try that next!

In [ ]:
for k in cache: 
    print(k, cache[k].shape)

In [ ]:
cache['hook_embed'].shape

In [ ]:
hook_embed_sq = einops.rearrange(cache['hook_embed'], "(x y) ... -> x y ...", x=p)

In [ ]:
hook_embed_sq.shape

In [ ]:
offset=48
num_points=p

fig=plt.figure(0, (12,12))
for i in range(12):
    for j in range(12):
        fig.add_subplot(12,12,12*i+j+1)
        plt.scatter(hook_embed_sq[:, 0, 0, i+offset].detach().cpu().numpy(), 
                    hook_embed_sq[0, :, 1, j+offset].detach().cpu().numpy(), 
                    c=range(num_points), s=1)
        plt.axis('off')

In [ ]:
plt.plot(cache['hook_embed'][:p, 1, 0].detach().cpu().numpy())

In [ ]:
cache['hook_embed'][:p, 0, 0].shape

In [ ]:
plt.imshow(cache['hook_embed'][:2*p, 0, :].detach().cpu().numpy())

In [ ]:
cache['hook_embed'].shape

In [ ]:
cache['hook_embed'][:10, :, :10]